### Лабораторная работа №2. Реализация метода главных компонент (PCA)

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
import plotly.graph_objects as go

In [2]:
data = load_breast_cancer()
data

{'data': array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
         1.189e-01],
        [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
         8.902e-02],
        [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
         8.758e-02],
        ...,
        [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
         7.820e-02],
        [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
         1.240e-01],
        [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
         7.039e-02]], shape=(569, 30)),
 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
        1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
        1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,

In [3]:
X = data.data
y = data.target

In [4]:
feature_names = data.feature_names

In [5]:
X.shape, y.shape

((569, 30), (569,))

In [6]:
data.target_names

array(['malignant', 'benign'], dtype='<U9')

In [7]:
X_df = pd.DataFrame(X, columns=feature_names)
X_df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [8]:
X_df.describe()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [9]:
def my_pca(X: np.ndarray, k: int):
    # Центрирование данных
    X_centered = X - X.mean(axis=0)
    # Построение ковариационной матрицы
    C = np.cov(X_centered, rowvar=False)
    # Диагонализация матрицы
    eigvals, eigvecs = np.linalg.eig(C)
    # Сортировка компонент
    sorted_indices = np.argsort(eigvals)[::-1]
    eigvals_sorted = eigvals[sorted_indices]
    eigvecs_sorted = eigvecs[:, sorted_indices]
    # Доля объясненной дисперсии
    explained_variance_ratio = eigvals_sorted / np.sum(eigvals_sorted)
    # Выбор K главных компонент
    W_K = eigvecs_sorted[:, :k]
    # Проекция данных
    Z = X_centered @ W_K
    # Возврат результата
    return Z, eigvals_sorted, explained_variance_ratio
    

In [10]:
Z2_manual, exp_variance_manual, exp_variance_ratio_manual = my_pca(X, 2)
f'{Z2_manual.shape=}'

'Z2_manual.shape=(569, 2)'

In [11]:
pca_sk = PCA(n_components=2)
Z2_sklearn, exp_variance_sklearn, exp_variance_ratio_sklearn = pca_sk.fit_transform(X), pca_sk.explained_variance_, pca_sk.explained_variance_ratio_
f'{Z2_sklearn.shape=}'

'Z2_sklearn.shape=(569, 2)'

In [18]:
print(f'{exp_variance_manual[:2]=}')
print(f'{exp_variance_sklearn=}')
print(f'{exp_variance_ratio_manual[:2]=}')
print(f'{exp_variance_ratio_sklearn=}')

exp_variance_manual[:2]=array([443782.6051466 ,   7310.10006165])
exp_variance_sklearn=array([443782.6051466 ,   7310.10006165])
exp_variance_ratio_manual[:2]=array([0.98204467, 0.01617649])
exp_variance_ratio_sklearn=array([0.98204467, 0.01617649])


In [21]:
components_2 = [1, 2]

bar_char_fit = go.Figure()
bar_char_fit.add_trace(
    go.Bar(name='Z2_manual', x=components_2, y=exp_variance_ratio_manual[:2])
)
bar_char_fit.add_trace(
    go.Bar(name='Z2_sklearn', x=components_2, y=exp_variance_ratio_sklearn)
)
bar_char_fit.update_layout(barmode='group')
bar_char_fit.show()

In [13]:
scatter_plot_fig = go.Figure()
scatter_plot_fig.add_trace(
    go.Scatter(
        x=Z2_manual[:, 0],
        y=Z2_manual[:, 1],
        mode='markers',
        name='Z2_manual'
    )
)
scatter_plot_fig.add_trace(
    go.Scatter(
        x=Z2_sklearn[:, 0],
        y=Z2_sklearn[:, 1],
        mode='markers',
        name='Z2_sklearn'
    )
)
scatter_plot_fig.show()

In [ ]:
components = [i for i in range(1, len(exp_variance_ratio_manual) + 1)]

scree_plot_fig = go.Figure()
scree_plot_fig.add_trace(
    go.Bar(name='bars', x=components, y=exp_variance_ratio_manual)
)
scree_plot_fig.add_trace(
    go.Scatter(x=components, y=exp_variance_ratio_manual, mode='lines+markers')
)
scree_plot_fig.show()

In [41]:
threshold = 0.999
cumulative = np.cumsum(exp_variance_ratio_manual)
threshold_line = [threshold for _ in range(len(exp_variance_ratio_manual))]

cumulative_fig = go.Figure()
cumulative_fig.add_trace(
    go.Scatter(name='aefea', x=components, y=cumulative, mode='lines+markers')
)
cumulative_fig.add_trace(
    go.Scatter(name='aefea', x=components, y=threshold_line, mode='lines')
)
cumulative_fig.show()

In [55]:
malignant_val = np.array([v for i, v in enumerate(Z2_manual) if y[i] == 0])
benign_val = np.array([v for i, v in enumerate(Z2_manual) if y[i] == 1])

class_scatter_plot_fig = go.Figure() 
class_scatter_plot_fig.add_trace(
    go.Scatter(
        x=malignant_val[:, 0],
        y=malignant_val[:, 1],
        mode='markers',
        name='malignant'
    )
)
class_scatter_plot_fig.add_trace(
    go.Scatter(
        x=benign_val[:, 0],
        y=benign_val[:, 1],
        mode='markers',
        name='benign'
    )
)
class_scatter_plot_fig.show()